# Causal Discovery with py-tetrad

This notebook demonstrates a range of causal discovery algorithms available in [py-tetrad](https://github.com/cmu-phil/py-tetrad), a Python interface to the [Tetrad](https://github.com/cmu-phil/tetrad) causal search library.

**Citation:**
> Ramsey, J., & Andrews, B. (2023, November). Py-Tetrad and RPy-Tetrad: A New Python Interface with R Support for Tetrad Causal Search. In *Causal Analysis Workshop Series* (pp. 40–51). PMLR.

---

## Setup

Before running this notebook, make sure the following prerequisites are in place:

1. **Java 21+** must be installed. See [Setting up Java for Tetrad](https://github.com/cmu-phil/tetrad/wiki/Setting-up-Java-for-Tetrad).
2. **`JAVA_HOME`** must be set to your JDK path. Check with:
   ```
   echo $JAVA_HOME
   ```
3. **Python 3.5+** is required.
4. Install dependencies:
   ```
   pip install JPype1
   pip install git+https://github.com/cmu-phil/py-tetrad
   ```
5. The dataset used here (`airfoil-self-noise.continuous.txt`) is available in the `pytetrad/resources/` directory of the cloned repository.

## Imports and Data Loading

In [133]:
import pandas as pd
import pytetrad.tools.TetradSearch as ts

# Load the airfoil self-noise dataset (continuous variables)
data = pd.read_csv("resources/airfoil-self-noise.continuous.txt", sep="\t")
data = data.astype({col: "float64" for col in data.columns})

print(f"Dataset shape: {data.shape}")
data.head()

Dataset shape: (1503, 6)


,Frequency,Attack,Chord,Velocity,Displacement,Pressure
0,800.0,0.0,0.3048,71.3,0.0027,126.201
1,1000.0,0.0,0.3048,71.3,0.0027,125.201
2,1250.0,0.0,0.3048,71.3,0.0027,125.951
3,1600.0,0.0,0.3048,71.3,0.0027,127.591
4,2000.0,0.0,0.3048,71.3,0.0027,127.461


## Initialize TetradSearch

`TetradSearch` is a convenience wrapper around Tetrad's Java algorithms. It manages the score, independence test, knowledge constraints, and algorithm execution, hiding the JPype machinery.

In [134]:
search = ts.TetradSearch(data)
search.set_verbose(True)

## Score and Test

We use the **SEM BIC score** (linear Gaussian) and the **Fisher Z test** (Gaussian CI test) as the default score and test throughout. Background knowledge is also set here: we forbid the edge `Frequency → Attack`.

In [135]:
search.use_sem_bic()
search.use_fisher_z(alpha=0.05)

# Background knowledge: Frequency cannot cause Attack
search.set_forbidden("Frequency", "Attack")

---
## CPDAG Algorithms

The following algorithms return **CPDAGs** (Completed Partially Directed Acyclic Graphs), representing the Markov equivalence class of a DAG under the assumption of causal sufficiency (no latent confounders).

### FGES (Fast Greedy Equivalence Search)

A score-based algorithm that searches greedily over equivalence classes.

In [136]:
print("FGES")
search.run_fges()
print(search.get_graph_to_matrix())

FGES
Elapsed initializeForwardEdgesFromEmptyGraph = 0 ms
1. INSERT Displacement --> Attack [] 1246.3174430377076 degree = 1 indegree = 1 cond = 1
2. INSERT Chord --> Attack [Displacement] 476.51847309066306 degree = 2 indegree = 1 cond = 2
--- Directing Displacement --> Attack
3. INSERT Frequency --> Pressure [] 234.33824857764193 degree = 2 indegree = 2 cond = 1
4. INSERT Displacement --> Pressure [Frequency] 324.6733710653607 degree = 2 indegree = 2 cond = 2
--- Directing Frequency --> Pressure
5. INSERT Chord --> Pressure [] 268.6067707995353 degree = 3 indegree = 3 cond = 3
6. INSERT Attack --> Frequency [] 101.57195639573547 degree = 3 indegree = 3 cond = 1
7. INSERT Velocity --> Pressure [] 90.11324718274045 degree = 4 indegree = 4 cond = 4
8. INSERT Attack --> Pressure [] 98.89198485187262 degree = 5 indegree = 5 cond = 5
9. INSERT Displacement --> Chord [] 60.45539122686023 degree = 5 indegree = 5 cond = 1
10. INSERT Velocity --> Frequency [] 22.332002679104335 degree = 5 indeg

### BOSS (Best Order Score Search)

A permutation-based score algorithm that tends to outperform FGES on many benchmarks. Also demonstrates extracting a DAG from the CPDAG and the adjacency matrix.

In [137]:
print("BOSS")
search.run_boss(num_starts=1, use_bes=True, time_lag=0, use_data_order=True)
print(search.get_string())

# Extract one DAG from the CPDAG
dag = search.get_dag_java()
print("\nA DAG in the equivalence class:")
print(dag)

# Adjacency matrix
print("\nAdjacency matrix:")
print(search.get_graph_to_matrix())

BOSS
Frequency
Attack
Chord
Velocity
Displacement
Pressure
Score: -46107.380
Velocity
Attack
Frequency
Displacement
Pressure
Chord
Score: -46107.380
Restart: 0	 Score: -46107.380	 Time: 0.001
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --- Displacement
3. Attack --> Frequency
4. Displacement --> Chord
5. Displacement --> Pressure
6. Frequency --> Chord
7. Frequency --> Pressure
8. Pressure --> Chord
9. Velocity --> Chord
10. Velocity --> Frequency
11. Velocity --> Pressure

Graph Attributes:
Score: -46107.380370
BIC: -46026.912968

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord: 2856.9619390766175;Velocity: -12518.376781557072;Displacement: 8815.0969551796;Pressure: -9054.27366892865]


A DAG in the equivalence class:
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --> Displacement
3. Attack --> Frequency
4. Disp

### SP (Sparsest Permutation)

Searches for the sparsest permutation of variables; exact but slower than BOSS for larger graphs.

In [138]:
print("SP")
search.run_sp()
print(search.get_string())

SP
SP: found 6 highest-scoring permutations (score=-46106.834126400194):
  Velocity , Chord , Displacement , Attack , Frequency , Pressure
  Velocity , Displacement , Chord , Attack , Frequency , Pressure
  Displacement , Velocity , Chord , Attack , Frequency , Pressure
  Displacement , Chord , Velocity , Attack , Frequency , Pressure
  Chord , Displacement , Velocity , Attack , Frequency , Pressure
  Chord , Velocity , Displacement , Attack , Frequency , Pressure
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Attack --> Pressure
3. Chord --> Attack
4. Chord --- Displacement
5. Chord --> Frequency
6. Chord --> Pressure
7. Displacement --> Attack
8. Displacement --> Pressure
9. Frequency --> Pressure
10. Velocity --> Attack
11. Velocity --> Frequency
12. Velocity --> Pressure

Graph Attributes:
Score: -46106.834126
BIC: -46019.051506

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882.65059720316;Chord:

### GRaSP (Greedy Relaxations of Sparsest Permutation)

A scalable relaxation of SP.

In [139]:
print("GRaSP")
search.run_grasp()
print(search.get_string())

GRaSP
Edges: 12 	|	 Score Improvement: 101.571956 	|	 Tucks Performed: [] [Frequency, Chord]
Edges: 11 	|	 Score Improvement: 13.956137 	|	 Tucks Performed: [] [Frequency, Velocity]
Edges: 11 	|	 Score Improvement: 4.025207 	|	 Tucks Performed: [] [Chord, Pressure]
Edges: 11 	|	 Score Improvement: 0.000000 	|	 Tucks Performed: [] [Attack, Displacement]
# Edges = 11 Score = -46107.38036982923 (GRaSP) Elapsed 0.0 s
Final order = [Displacement, Attack, Velocity, Frequency, Pressure, Chord]
Elapsed time = 0.0 s
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --> Frequency
3. Displacement --- Attack
4. Displacement --> Chord
5. Displacement --> Pressure
6. Frequency --> Chord
7. Frequency --> Pressure
8. Pressure --> Chord
9. Velocity --> Chord
10. Velocity --> Frequency
11. Velocity --> Pressure

Graph Attributes:
Score: -46107.380370
BIC: -46026.912968

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -7882

### PC, PC-Max, and CPC

Constraint-based algorithms using the Fisher Z independence test. PC is the classic Peter-Clark algorithm; PC-Max and CPC are variants with different collider orientation strategies.

In [140]:
print("PC")
search.run_pc()
print(search.get_string())

PC
Depth: 0
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Displacement _||_ Velocity	p = 0.8726
Independence accepted: Chord _||_ Velocity	p = 0.8834
Depth: 1
Independence accepted: Displacement _||_ Frequency | Attack	p = 0.1306
Depth: 2
Depth: 3
Independence accepted: Attack _||_ Pressure | Frequency, Velocity, Displacement	p = 0.2164
Depth: 4
Independence accepted: Chord _||_ Frequency | Velocity	p = 0.8707
Independence accepted: Chord _||_ Velocity | Attack	p = 0.1326
Independence accepted: Chord _||_ Velocity | Displacement	p = 0.9092
Independence accepted: Chord _||_ Velocity | Pressure	p = 0.1805
Independence accepted: Chord _||_ Velocity | Frequency	p = 0.8673
Independence accepted: Chord _||_ Velocity | Pressure, Displacement	p = 0.0576
Independence accepted: Displacement _||_ Frequency | Attack, Chord	p = 0.7161
Independence accepted: Displacement _||_ Frequency | Attack, Velocity	p = 0.2794
Independence accepted: Displacement _||_ Velocity | C

In [141]:
print("PC-Max")
search.run_pc_max()
print(search.get_string())

PC-Max
Depth: 0
Independence accepted: Displacement _||_ Velocity	p = 0.8726
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Chord _||_ Velocity	p = 0.8834
Depth: 1
Independence accepted: Displacement _||_ Frequency | Attack	p = 0.1306
Depth: 2
Depth: 3
Independence accepted: Attack _||_ Pressure | Frequency, Velocity, Displacement	p = 0.2164
Depth: 4
Independence accepted: Chord _||_ Frequency | Velocity	p = 0.8707
Independence accepted: Chord _||_ Velocity | Attack	p = 0.1326
Independence accepted: Chord _||_ Velocity | Displacement	p = 0.9092
Independence accepted: Chord _||_ Velocity | Pressure	p = 0.1805
Independence accepted: Chord _||_ Velocity | Frequency	p = 0.8673
Independence accepted: Chord _||_ Velocity | Pressure, Displacement	p = 0.0576
Ambiguous triple: Chord - Attack - Velocity
Ambiguous triple: Chord - Pressure - Velocity
Independence accepted: Displacement _||_ Frequency | Attack, Chord	p = 0.7161
Independence accepted: Displacement _||_

In [142]:
print("CPC (Conservative PC)")
search.run_cpc()
print(search.get_string())

CPC (Conservative PC)
Depth: 0
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Displacement _||_ Velocity	p = 0.8726
Independence accepted: Chord _||_ Velocity	p = 0.8834
Depth: 1
Independence accepted: Displacement _||_ Frequency | Attack	p = 0.1306
Depth: 2
Depth: 3
Independence accepted: Attack _||_ Pressure | Frequency, Velocity, Displacement	p = 0.2164
Depth: 4
Independence accepted: Chord _||_ Frequency | Velocity	p = 0.8707
Independence accepted: Chord _||_ Velocity | Attack	p = 0.1326
Independence accepted: Chord _||_ Velocity | Displacement	p = 0.9092
Independence accepted: Chord _||_ Velocity | Pressure	p = 0.1805
Independence accepted: Chord _||_ Velocity | Frequency	p = 0.8673
Independence accepted: Chord _||_ Velocity | Pressure, Displacement	p = 0.0576
Independence accepted: Displacement _||_ Frequency | Attack, Chord	p = 0.7161
Independence accepted: Displacement _||_ Frequency | Attack, Velocity	p = 0.2794
Independence accepted: Displacemen

---
## PAG Algorithms

The following algorithms return **PAGs** (Partial Ancestral Graphs), which represent Markov equivalence classes of MAGs and allow for the possibility of latent confounders and/or selection bias.

### FCI (Fast Causal Inference)

In [143]:
print("FCI")
search.run_fci()
print(search.get_string())

FCI
Starting FCI algorithm.
Independence test = edu.cmu.tetrad.search.test.CachingIndependenceTest@5a537a16.
Starting FAS search.
Depth: 0
Independence accepted: Displacement _||_ Velocity	p = 0.8726
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Chord _||_ Velocity	p = 0.8834
Depth: 1
Independence accepted: Displacement _||_ Frequency | Attack	p = 0.1306
Depth: 2
Depth: 3
Independence accepted: Attack _||_ Pressure | Frequency, Velocity, Displacement	p = 0.2164
Depth: 4
Reorienting with o-o.
Applying R0 (MAX_P).
Starting BK Orientation.
Finishing BK Orientation.
[R0-MAX_P] Attack -> Chord <- Pressure
Independence accepted: Chord _||_ Frequency | Velocity	p = 0.8707
Independence accepted: Chord _||_ Velocity | Attack	p = 0.1326
Independence accepted: Chord _||_ Velocity | Displacement	p = 0.9092
Independence accepted: Chord _||_ Velocity | Pressure	p = 0.1805
Independence accepted: Chord _||_ Velocity | Frequency	p = 0.8673
Independence accepted: Chord _|

### GFCI (Greedy FCI)

Combines FGES with FCI for improved performance.

In [144]:
print("GFCI")
search.run_gfci()
print(search.get_string())

GFCI
Starting FGES.
Elapsed initializeForwardEdgesFromEmptyGraph = 0 ms
1. INSERT Displacement --> Attack [] 1246.3174430377076 degree = 1 indegree = 1 cond = 1
2. INSERT Chord --> Attack [Displacement] 476.51847309066306 degree = 2 indegree = 1 cond = 2
--- Directing Displacement --> Attack
3. INSERT Frequency --> Pressure [] 234.33824857764193 degree = 2 indegree = 2 cond = 1
4. INSERT Displacement --> Pressure [Frequency] 324.6733710653607 degree = 2 indegree = 2 cond = 2
--- Directing Frequency --> Pressure
5. INSERT Chord --> Pressure [] 268.6067707995353 degree = 3 indegree = 3 cond = 3
6. INSERT Attack --> Frequency [] 101.57195639573547 degree = 3 indegree = 3 cond = 1
7. INSERT Velocity --> Pressure [] 90.11324718274045 degree = 4 indegree = 4 cond = 4
8. INSERT Attack --> Pressure [] 98.89198485187262 degree = 5 indegree = 5 cond = 5
9. INSERT Displacement --> Chord [] 60.45539122686023 degree = 5 indegree = 5 cond = 1
10. INSERT Velocity --> Frequency [] 22.332002679104335 d

### FCIT (FCI with Targeted Testing)

FCIT is a hybrid PAG search algorithm that combines an FCI-style procedure with targeted conditional independence testing. It uses BOSS as a CPDAG oracle to guide the search, aiming to improve efficiency while still allowing for latent confounders.


In [145]:
print("FCIT")
search.run_fcit()
print(search.get_string())

FCIT
===Starting FCIT===

Round: 1

Removing Velocity o-> Chord for recursive reasons, sepset = [Pressure, Displacement]
Removing Chord <-> Frequency for recursive reasons, sepset = [Velocity]

Round: 2


FCIT finished.
BOSS/GRaSP time: 1 ms.
Collider orientation and edge removal time: 3 ms.
Total time: 4 ms.
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Displacement
2. Chord <-> Attack
3. Chord <-> Pressure
4. Displacement <-> Chord
5. Displacement --> Pressure
6. Frequency <-> Attack
7. Frequency --> Pressure
8. Velocity o-> Frequency
9. Velocity --> Pressure

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -9610.057963361405;Chord: 3284.553032306396;Velocity: -12518.376781557072;Displacement: 10110.341778078193;Pressure: -9054.27366892865]



### GRaSP-FCI

In [146]:
print("GRaSP-FCI")
search.run_grasp_fci()
print(search.get_string())

GRaSP-FCI
Starting GRaSP.
Edges: 12 	|	 Score Improvement: 101.571956 	|	 Tucks Performed: [] [Frequency, Chord]
Edges: 11 	|	 Score Improvement: 9.432070 	|	 Tucks Performed: [[Frequency, Chord]] [Frequency, Velocity]
Edges: 11 	|	 Score Improvement: 4.524068 	|	 Tucks Performed: [] [Frequency, Chord]
Edges: 11 	|	 Score Improvement: 4.025207 	|	 Tucks Performed: [] [Chord, Pressure]
Edges: 11 	|	 Score Improvement: 0.000000 	|	 Tucks Performed: [] [Attack, Displacement]
# Edges = 11 Score = -46107.38036982923 (GRaSP) Elapsed 0.001 s
Final order = [Displacement, Attack, Velocity, Frequency, Pressure, Chord]
Elapsed time = 0.001 s
Finished GRaSP.
Starting *-FCI extra edge removal step.
Independence accepted: Frequency _||_ Chord | Velocity	p = 0.8707
Independence accepted: Frequency _||_ Chord	p = 0.8873
Removed edge Frequency -- Chord in extra-edge removal step; sepset = [], p-value = 0.8872564937615057.
Independence accepted: Velocity _||_ Chord | Pressure	p = 0.1805
Independence acc

### BOSS-FCI

In [147]:
print("BOSS-FCI")
search.run_boss_fci()
print(search.get_string())

BOSS-FCI
Starting BOSS.
Frequency
Attack
Chord
Velocity
Displacement
Pressure
Score: -46107.380
Velocity
Attack
Frequency
Displacement
Pressure
Chord
Score: -46107.380
Restart: 0	 Score: -46107.380	 Time: 0.001
Finished BOSS.
Starting *-FCI extra edge removal step.
Independence accepted: Frequency _||_ Chord | Velocity	p = 0.8707
Independence accepted: Frequency _||_ Chord	p = 0.8873
Independence accepted: Chord _||_ Frequency | Velocity	p = 0.8707
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Frequency _||_ Chord	p = 0.8873
Independence accepted: Frequency _||_ Chord	p = 0.8873
Independence accepted: Frequency _||_ Chord	p = 0.8873
Removed edge Frequency -- Chord in extra-edge removal step; sepset = [], p-value = 0.8872564937615057.
Independence accepted: Velocity _||_ Chord | Pressure	p = 0.1805
Independence accepted: Velocity _||_ Chord	p = 0.8834
Independence accepted: Velocity _||_ Chord | Frequency	p = 0.8673
Independence accepted: Chord _||_ Veloc

### LV-Heuristic

In [148]:
print("LV-Heuristic")
search.run_lv_heuristic()
print(search.get_string())

LV-Heuristic
===Starting LV-Heuristic===
Starting BOSS.
Frequency
Attack
Chord
Velocity
Displacement
Pressure
Score: -46107.380
Attack
Velocity
Frequency
Displacement
Pressure
Chord
Score: -46107.380
Restart: 0	 Score: -46107.380	 Time: 0.000
Finished BOSS.
Calculating PAG from DAG.
Finished calculating PAG from CPDAG.
LV-Heuristic finished.
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack o-> Chord
2. Attack o-o Displacement
3. Attack o-> Frequency
4. Displacement o-> Chord
5. Displacement --> Pressure
6. Frequency o-> Chord
7. Frequency --> Pressure
8. Pressure o-> Chord
9. Velocity o-> Chord
10. Velocity o-> Frequency
11. Velocity --> Pressure

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -9610.057963361405;Chord: 3284.553032306396;Velocity: -12518.376781557072;Displacement: 10110.341778078193;Pressure: -9054.27366892865]



### CCD (Cyclic Causal Discovery)

Handles cyclic graphs; does not use background knowledge.

In [149]:
print("CCD")
search.run_ccd()
print(search.get_string())

CCD
CCD does not use knowledge.
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack o-> Chord
2. Attack o-o Displacement
3. Attack o-> Frequency
4. Displacement o-> Chord
5. Displacement --> Pressure
6. Frequency o-> Chord
7. Frequency --> Pressure
8. Pressure o-> Chord
9. Velocity o-> Chord
10. Velocity o-> Frequency
11. Velocity --> Pressure

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -9610.057963361405;Chord: 3284.553032306396;Velocity: -12518.376781557072;Displacement: 10110.341778078193;Pressure: -9054.27366892865]



### SP-FCI

In [150]:
print("SP-FCI")
search.run_sp_fci()
print(search.get_string())

SP-FCI
Starting SP.
SP: found 6 highest-scoring permutations (score=-46106.834126400194):
  Velocity , Chord , Displacement , Attack , Frequency , Pressure
  Velocity , Displacement , Chord , Attack , Frequency , Pressure
  Displacement , Velocity , Chord , Attack , Frequency , Pressure
  Displacement , Chord , Velocity , Attack , Frequency , Pressure
  Chord , Displacement , Velocity , Attack , Frequency , Pressure
  Chord , Velocity , Displacement , Attack , Frequency , Pressure
Finished SP.
Starting *-FCI extra edge removal step.
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Frequency _||_ Chord	p = 0.8873
Independence accepted: Frequency _||_ Chord | Velocity	p = 0.8707
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Chord _||_ Frequency	p = 0.8873
Independence accepted: Chord _||_ Frequency	p = 0.8873
Removed edge Chord -- Frequency in extra-edge removal step; sepset = [], p-value = 0.8872564937615057.
Independence acce

---
## DAG Algorithms

The following algorithms return fully oriented **DAGs** under additional assumptions (e.g., non-Gaussianity or nonlinearity).

### LiNGAM (Linear Non-Gaussian Acyclic Model)

Exploits non-Gaussianity of noise to identify a unique DAG.

In [151]:
print("LiNGAM")
search.run_lingam(threshold_b=0.06)
print(search.get_string())
print("\nB-hat matrix (estimated coefficients):")
print(search.get_bhat())

LiNGAM
Anderson Darling P-values Per Variables (p < alpha means Non-Gaussian)

Frequency: p = 0.000
Attack: p = 0.000
Chord: p = 0.000
Velocity: p = 0.000
Displacement: p = 0.000
Pressure: p = 0.000
LiNGAM: effective trim threshold = 0.07122522016695942
BHat = 
  V1        V2          V3        V4           V5         V6
       23.7019  -2468.3746   25.2326  -55773.2183  -154.5269
                  -24.5688               158.9256           
                                           0.9180           
        0.1920      8.3534                 1.4679     0.2006
                                                            
       -0.3179    -48.7516              -224.2628           

Graph = Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Attack --> Pressure
3. Attack --> Velocity
4. Chord --> Attack
5. Chord --> Frequency
6. Chord --> Pressure
7. Chord --> Velocity
8. Displacement --> Attack
9. Displacement --> Chord
10. Displac

### FASK (Fast Adjacency Skewness)

Uses skewness of residuals to orient edges.

In [152]:
print("FASK")
search.run_fask()
print(search.get_string())

FASK
Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Chord
2. Attack --> Frequency
3. Chord --> Displacement
4. Chord --> Pressure
5. Displacement --> Attack
6. Displacement --> Pressure
7. Frequency --> Velocity
8. Pressure --> Frequency
9. Velocity --> Pressure

Graph Node Attributes:
Score: [Frequency: -28323.59197296753;Attack: -9610.057963361405;Chord: 3284.553032306396;Velocity: -12518.376781557072;Displacement: 10110.341778078193;Pressure: -9054.27366892865]



### LiNG-D (Linear Non-Gaussian with Dependence)

An extension of LiNGAM that handles cyclic models and returns potentially multiple stable B-hat matrices.

In [153]:
print("LiNG-D")
search.set_verbose(False)
search.run_lingd(threshold_w=1e-4)

print("\nUnstable B-hats:")
print(search.get_unstable_bhats())

print("\nStable B-hats:")
print(search.get_stable_bhats())

LiNG-D
To see unstable models and and their B matrices, set the verbose flag to true
LiNG-D Model #1 Stable = True

  V1          V2           V3        V4            V5         V6
      -1343.2406  -45578.9589   80.1168   119073.6616  -440.1400
                      -8.6134                795.5729           
                                                                
                       1.7147                  3.9023           
                                                                
          0.2168     -20.0528               -218.0195           

Graph Nodes:
Frequency;Attack;Chord;Velocity;Displacement;Pressure

Graph Edges:
1. Attack --> Frequency
2. Attack --> Pressure
3. Chord --> Attack
4. Chord --> Frequency
5. Chord --> Pressure
6. Chord --> Velocity
7. Displacement --> Attack
8. Displacement --> Frequency
9. Displacement --> Pressure
10. Displacement --> Velocity
11. Pressure --> Frequency
12. Velocity --> Frequency

Graph Node Attributes:
Score: [Frequency: 

---
## Notes

- All algorithms above use the SEM BIC score and Fisher Z test set at the top of this notebook. You can substitute other scores and tests (e.g., `use_ebic()`, `use_kci()`, `use_ffml()`, `use_ffci()`) depending on your assumptions about the data-generating process.
- For **nonlinear data**, consider replacing `use_sem_bic()` with `use_ffml()` and `use_fisher_z()` with `use_ffci()`. See the [FFML/TRFF/FFCI paper](https://arxiv.org/abs/2605.05743) for details.
- For **time series data**, see the `run_svar_fci()` method and the `TsUtils` utilities in Tetrad.
- Full API documentation is available at the [Tetrad ReadTheDocs manual](https://tetrad-manual.readthedocs.io/en/latest/).